In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from net import UpscalerNet
from dataset import SuperResHsvDataset
from lightning_net import LitUpscaler, ImageLoggerCallback
import matplotlib.pyplot as plt


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [2]:
images = []
lesions = []
from skimage.io import imread
import os


train_dataset = SuperResHsvDataset(
    r'C:\Users\efimplotnikov\Pictures\2016 - велики\Осень 2016', 
    crop_size=64, 
    downscale_denoise=2
)
val_dataset = SuperResHsvDataset(
    r'C:\Users\efimplotnikov\Pictures\2016 - велики\Лето 2016\07.23 - 07.24 - вырица всей толпой, грибы', 
    crop_size=64, 
    downscale_denoise=2
)

print(f'Loaded {len(train_dataset)} train mages, {len(val_dataset)} val images')

Loaded 698 train mages, 121 val images


In [3]:
import torch
import torch.nn as nn

In [4]:
def eval_using_cp(cp, model, X):
    checkpoint = torch.load(cp, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    with torch.no_grad():
        return model(X.to(DEVICE)).cpu()

In [5]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs --port 6006


Reusing TensorBoard on port 6006 (pid 12148), started 4 days, 8:27:22 ago. (Use '!kill 12148' to kill it.)

In [ ]:
import lightning as L

from lightning.pytorch.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

from loss import HSVPerceptualLoss, HSVSimpleLoss

torch.set_float32_matmul_precision('medium')

checkpoint_cb = ModelCheckpoint(
    dirpath="checkpoints",
    filename="upscaler-{epoch:03d}",
    save_top_k=3,
    monitor="val/loss",
    mode="min"
)

val_dl_for_logging = DataLoader(val_dataset, batch_size=8, shuffle=False)
logger_cb = ImageLoggerCallback(val_dl_for_logging, log_every_n_epochs=1)

loss = HSVSimpleLoss(hue_weight=20, s_weight=10, v_weight=30)

upscaler = UpscalerNet(num_features=128).to(DEVICE)
lit = LitUpscaler(upscaler, lr=1e-4)
lit.set_loss(loss)

trainer = L.Trainer(
    max_epochs=600,
    logger = TensorBoardLogger("lightning_logs", name="version_3", version=3),
    log_every_n_steps=5,
    callbacks=[checkpoint_cb, logger_cb],
)

train_dl = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=8, persistent_workers=True)
val_dl   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=8, persistent_workers=True)
trainer.fit(lit, train_dl, val_dl)

c:\Users\efimplotnikov\Projects\DeepLearningSchool_part_1_ml_cv\upscaler\dataset.py:56: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  arr = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes()))
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type          | Params | Mode  | FLOPs
--------------------------------------------------------
0 | model | UpscalerNet   | 310 M  | train | 0    
1 | loss  | HSVSimpleLoss | 0      | train | 0    
--------------------------------------------------------
310 M     Trainable params
0         Non-trainable params
310 M     Total params
1,240.672 Total estimated model params size (MB)
105       Modules in train mode
0   

LR inside configure: 0.0001


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=600` reached.


In [7]:
upscaler_test = LitUpscaler.load_from_checkpoint(
    'checkpoints/upscaler-epoch=468-loss=0.0000.ckpt',
    model=UpscalerNet(num_features=32)
    ).to(DEVICE)

upscaler_test.eval()

FileNotFoundError: [Errno 2] No such file or directory: 'c:/Users/efimplotnikov/Projects/DeepLearningSchool_part_1_ml_cv/upscaler/checkpoints/upscaler-epoch=468-loss=0.0000.ckpt'

In [ ]:
test_dataset = SuperResHsvDataset(r'C:\Users\efimplotnikov\Pictures\2016 - велики\Лето 2016', downscale=2, crop_size=64)


test_loader = DataLoader(test_dataset,
                     batch_size=8, shuffle=False)

for X, y_true in test_loader:
    with torch.no_grad():
        y_pred = upscaler_test(X.to(DEVICE)).cpu()

    for i in range(len(X)):
        
        fig, ax = plt.subplots(1, 3, figsize=(15,5))

        downscaled_x = test_dataset.hsv_tensor_to_pil(X[i])
        ax[0].imshow(downscaled_x)
        ax[0].set_title("Low-res (X)")
        ax[0].axis("off")

        ax[1].imshow(test_dataset.hsv_tensor_to_pil(y_true[i]))
        ax[1].set_title("HR Ground Truth (Y)")
        ax[1].axis("off")

        ax[2].imshow(test_dataset.hsv_tensor_to_pil(y_pred[i]))
        ax[2].set_title("Prediction")
        ax[2].axis("off") 

        plt.show()
    break
